# Квантизация нейронных сетей

## Подробный учебник по теории и практике

*От математических основ до современных методов LLM-квантизации*

---

## Содержание

1. [Введение в квантизацию](#глава-1-введение-в-квантизацию)
2. [Математические основы](#глава-2-математические-основы)
3. [Типы числовых представлений](#глава-3-типы-числовых-представлений)
4. [Схемы квантизации](#глава-4-схемы-квантизации)
5. [Post-Training Quantization (PTQ)](#глава-5-post-training-quantization-ptq)
6. [Quantization-Aware Training (QAT)](#глава-6-quantization-aware-training-qat)
7. [Продвинутые методы для LLM](#глава-7-продвинутые-методы-для-llm)
8. [Квантизация активаций и KV-кэша](#глава-8-квантизация-активаций-и-kv-кэша)
9. [Аппаратные аспекты и форматы](#глава-9-аппаратные-аспекты-и-форматы)
10. [Практические рекомендации](#глава-10-практические-рекомендации)
11. [Приложение А. Глоссарий](#приложение-а-глоссарий)
12. [Приложение Б. Литература и ресурсы](#приложение-б-литература-и-ресурсы)

---

## Глава 1. Введение в квантизацию

### 1.1. Что такое квантизация?

**Квантизация нейронных сетей** — это процесс уменьшения разрядности (числа битов) для представления весов и/или активаций модели. Если исходная сеть оперирует 32-битными числами с плавающей точкой (FP32), то после квантизации она может работать с 8-битными целыми числами (INT8), 4-битными (INT4) или даже более компактными форматами.

Если упростить: квантизация — это «округление» бесконечно точных вещественных весов до конечного набора дискретных значений. Подобно тому, как цифровая фотография аппроксимирует непрерывный спектр цветов набором из 256 градаций на канал, квантизация аппроксимирует пространство весов сеткой допустимых значений.

Формально, квантизация — это отображение:

$$Q: \mathbb{R} \to \mathcal{C}, \quad |\mathcal{C}| = 2^b$$

где $\mathcal{C}$ — конечное множество кодов мощностью $2^b$, а $b$ — число битов на значение.

### 1.2. Зачем квантизовать модели?

Существует четыре основные причины применять квантизацию:

1. **Сокращение объёма памяти.** Модель с 7 миллиардами параметров в FP16 занимает ~14 ГБ, в INT8 — ~7 ГБ, а в INT4 — всего ~3.5 ГБ. Это позволяет запускать большие модели на потребительских GPU.
2. **Ускорение вычислений.** Современные процессоры (GPU, TPU, NPU) выполняют целочисленные операции значительно быстрее операций с плавающей точкой. Например, NVIDIA Tensor Cores дают 2× ускорение для INT8 относительно FP16.
3. **Снижение энергопотребления.** Целочисленные операции потребляют в 3–10 раз меньше энергии, что критично для мобильных и edge-устройств.
4. **Уменьшение пропускной способности памяти.** Поскольку инференс LLM часто упирается в memory bandwidth, уменьшение размера весов даёт прямое ускорение даже без изменения скорости вычислений.

> 💡 **Важно:** квантизация — это компромисс между эффективностью и точностью. Качественные методы квантизации минимизируют потерю точности, но никогда не устраняют её полностью.

### 1.3. Краткая история

Квантизация известна в обработке сигналов с середины XX века, но в контексте глубокого обучения её систематическое применение началось в 2015–2016 годах. Ключевые вехи:

- **2015** — Han et al. предложили *Deep Compression* (комбинация прунинга, квантизации и кодирования Хаффмана).
- **2018** — Jacob et al. (Google) формализуют INT8-квантизацию для мобильного инференса в TensorFlow Lite.
- **2019** — появление QAT (Quantization-Aware Training) как стандартного подхода.
- **2022** — GPTQ и LLM.int8() открывают эпоху эффективной квантизации больших языковых моделей.
- **2023** — AWQ, SmoothQuant, QLoRA делают возможным запуск 70B-моделей на одной GPU.
- **2024–2025** — развитие 4-битных и даже 2-битных форматов (например, BitNet b1.58, GGUF Q2_K).

---

## Глава 2. Математические основы

### 2.1. Аффинная (asymmetric) квантизация

Наиболее общая формула квантизации связывает вещественное число $r$ с целым $q$ через два параметра: **масштаб** (scale) $s$ и **нулевую точку** (zero-point) $z$.

$$q = \mathrm{round}\!\left(\frac{r}{s}\right) + z$$

$$\hat{r} = s \cdot (q - z)$$

Здесь:
- $s \in \mathbb{R}_{>0}$ — положительное вещественное число, определяющее шаг сетки;
- $z \in \mathbb{Z}$ — целое число, обеспечивающее точное представление нуля.

Нулевая точка важна, потому что в нейросетях значение $0$ встречается часто (паддинг, ReLU, маски) и не должно искажаться.

#### Вычисление параметров

Для диапазона исходных значений $[r_{\min}, r_{\max}]$ и целевого диапазона целых чисел $[q_{\min}, q_{\max}]$ (например, $[-128, 127]$ для INT8):

$$s = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}}$$

$$z = \mathrm{round}\!\left(q_{\min} - \frac{r_{\min}}{s}\right)$$

### 2.2. Симметричная квантизация

Если нулевая точка $z = 0$, квантизация называется **симметричной**. Тогда диапазон значений симметричен относительно нуля: $[-r_{\max}, r_{\max}]$.

$$q = \mathrm{round}\!\left(\frac{r}{s}\right), \quad s = \frac{r_{\max}}{q_{\max}}$$

Симметричная квантизация проще и быстрее на этапе инференса (не нужно вычитать $z$), но менее эффективна при сильно асимметричных распределениях, типичных для активаций после ReLU (где все значения неотрицательны).

| Свойство | Симметричная | Асимметричная |
|---|---|---|
| Параметры | Только $s$ | $s$ и $z$ |
| Точное представление 0 | Да | Да |
| Использование диапазона | Неэффективно для несимметричных данных | Максимально полное |
| Сложность инференса | Меньше (нет вычитания $z$) | Чуть выше |
| Типичное применение | Веса | Активации |

### 2.3. Ошибка квантизации

Каждое квантованное значение отличается от исходного на некоторую величину — это и есть ошибка квантизации:

$$e(r) = r - \hat{r} = r - s \cdot \mathrm{round}\!\left(\frac{r}{s}\right)$$

Если распределение значений равномерно в пределах одного шага сетки, ошибка распределена равномерно на $[-s/2, s/2]$, а её математическое ожидание квадрата (дисперсия) равно:

$$\mathbb{E}[e^2] = \frac{s^2}{12}$$

Эта величина — фундаментальный предел точности равномерной квантизации. Любая квантизационная схема должна стремиться минимизировать ожидаемую ошибку при ограничении на число битов.

Для тензора весов $W \in \mathbb{R}^{m \times n}$ суммарная ошибка реконструкции:

$$\mathcal{L}_{\text{quant}} = \| W - \hat{W} \|_F^2 = \sum_{i,j} (W_{ij} - \hat{W}_{ij})^2$$

где $\| \cdot \|_F$ — норма Фробениуса.

### 2.4. Калибровка: выбор диапазона

Главный практический вопрос: как выбрать $r_{\min}$ и $r_{\max}$? Простой подход — взять минимум и максимум по тензору, но это уязвимо к выбросам (outliers): несколько экстремальных значений могут раздуть диапазон и сделать сетку слишком грубой для основной массы данных.

#### Методы калибровки

- **Min-Max** — простейший: $[r_{\min}, r_{\max}] = [\min(X), \max(X)]$. Чувствителен к выбросам.
- **Percentile** — берётся диапазон между, например, $0.01$-м и $99.99$-м перцентилями. Устойчивее к выбросам.
- **MSE-минимизация** — выбираются такие $r_{\min}, r_{\max}$, которые минимизируют:

$$\min_{r_{\min}, r_{\max}} \mathbb{E}_{x \sim \mathcal{D}}\!\left[ \big(x - Q(x; r_{\min}, r_{\max})\big)^2 \right]$$

- **KL-divergence (Entropy)** — диапазон подбирается так, чтобы минимизировать расхождение Кульбака–Лейблера между распределениями:

$$D_{\mathrm{KL}}(P \,\|\, Q) = \sum_i P_i \log \frac{P_i}{Q_i}$$

где $P$ — гистограмма оригинальных значений, $Q$ — квантованных. Применяется в TensorRT.

- **Cross-Entropy** — оптимизация диапазона по итоговой задаче (например, классификация).

> 💡 **На практике:** для весов часто хватает Min-Max. Для активаций обычно нужны более продвинутые методы калибровки на репрезентативной выборке.

---

## Глава 3. Типы числовых представлений

### 3.1. Форматы с плавающей точкой

Число с плавающей точкой представляется в виде:

$$x = (-1)^s \cdot 2^{e - \text{bias}} \cdot (1.m)_2$$

где $s$ — знак, $e$ — экспонента, $m$ — мантисса.

| Формат | Биты | Экспонента | Мантисса | Особенности |
|---|---|---|---|---|
| FP32 (single) | 32 | 8 | 23 | Стандарт IEEE 754; baseline для обучения |
| FP16 (half) | 16 | 5 | 10 | Узкий диапазон; склонен к overflow |
| BF16 (bfloat16) | 16 | 8 | 7 | Тот же диапазон, что FP32; меньше точности |
| FP8 (E4M3) | 8 | 4 | 3 | Для весов; точнее, но уже диапазон |
| FP8 (E5M2) | 8 | 5 | 2 | Для градиентов; шире диапазон |
| FP4 / NF4 | 4 | — | — | Применяется в QLoRA; кодовая таблица |

**BF16 (Brain Float 16)** — формат, разработанный Google, особенно популярен в обучении больших моделей. Он имеет тот же диапазон, что FP32 (важно для предотвращения overflow при backward pass), но меньшую точность мантиссы. Это компромисс, который оказался удачным для DL.

Диапазон BF16:

$$|x|_{\max} \approx 3.39 \times 10^{38}, \quad |x|_{\min} \approx 1.18 \times 10^{-38}$$

### 3.2. Целочисленные форматы

- **INT8** — диапазон $[-128, 127]$ (signed) или $[0, 255]$ (unsigned). Главный «рабочий» формат для инференса.
- **INT4** — диапазон $[-8, 7]$ (signed) или $[0, 15]$ (unsigned). Используется для весов LLM, агрессивно сжимает модель в 4 раза относительно FP16.
- **INT2 / Ternary** — экспериментальные форматы (например, BitNet b1.58 использует $\{-1, 0, +1\}$).
- **Binary (1-bit)** — крайний случай: каждый вес представлен одним битом. Подходит лишь для специализированных архитектур (XNOR-Net, BinaryConnect).

### 3.3. Специальные форматы для LLM

#### NF4 (NormalFloat 4)

Формат, предложенный в работе *QLoRA* (Dettmers et al., 2023). Идея: веса нейросетей часто распределены нормально (по Гауссу), $W_{ij} \sim \mathcal{N}(0, \sigma^2)$, поэтому равномерная сетка из 16 значений неоптимальна. NF4 использует **неравномерную** сетку, где значения $\{q_i\}_{i=0}^{15}$ подобраны так, чтобы соответствовать перцентилям стандартного нормального распределения:

$$q_i = \frac{1}{2}\left( \Phi^{-1}\!\left(\frac{i + 0.5}{16}\right) + \Phi^{-1}\!\left(\frac{i + 1.5}{16}\right) \right)$$

где $\Phi^{-1}$ — обратная функция распределения $\mathcal{N}(0,1)$. Это даёт лучшую точность, чем INT4 или FP4 при той же разрядности.

#### GGUF (Q2_K, Q3_K, Q4_K, Q5_K, Q6_K, Q8_0)

Семейство форматов из проекта *llama.cpp*. Использует блочную квантизацию с разным уровнем точности для разных частей модели. Суффикс `_K` означает «k-quants» — улучшенный метод, при котором scale и zero-point также квантуются. Например, Q4_K_M даёт ~4.5 бит на параметр в среднем при минимальной потере качества.

---

## Глава 4. Схемы квантизации

### 4.1. Гранулярность

Один из ключевых дизайн-параметров — на каком уровне применяются параметры квантизации $(s, z)$.

| Уровень | Описание | Trade-off |
|---|---|---|
| Per-tensor | Один $s$ и $z$ на весь тензор (например, на всю матрицу весов слоя) | Минимум overhead, максимум потери точности |
| Per-channel | Свои $s_i, z_i$ для каждого выходного канала | Стандарт для весов CNN и линейных слоёв |
| Per-token | Для активаций: свои параметры на каждый токен | Снижает влияние outliers в активациях |
| Per-group | Группы по 32, 64 или 128 элементов имеют свои параметры | Компромисс точность/размер; стандарт для INT4 LLM |

Для **per-group квантизации** с размером группы $g$ матрица весов разбивается на блоки:

$$W = \begin{bmatrix} W^{(1)} \\ W^{(2)} \\ \vdots \\ W^{(K)} \end{bmatrix}, \quad K = \frac{n}{g}$$

и каждый блок $W^{(k)}$ квантуется со своими $(s_k, z_k)$.

> 💡 **На практике:** для LLM в INT4 повсеместно используется per-group квантизация с группами по 128 элементов: это хорошо балансирует точность и накладные расходы на хранение параметров квантизации.

### 4.2. Статическая и динамическая квантизация

#### Статическая квантизация

Параметры квантизации $(s, z)$ для активаций **вычисляются заранее** на калибровочном наборе данных и фиксируются. На инференсе они используются как константы.

- **Плюсы:** максимальная скорость инференса, никаких runtime-вычислений.
- **Минусы:** качество зависит от репрезентативности калибровочного набора; чувствительна к outlier-активациям.

#### Динамическая квантизация

Параметры активаций **вычисляются на лету** для каждого входа.

- **Плюсы:** всегда оптимальный диапазон для текущего входа; не нужна калибровка.
- **Минусы:** дополнительные вычисления на каждой итерации; сложнее эффективно реализовать на GPU.

### 4.3. Симметричный против асимметричного выбора знака

На практике сложилась следующая конвенция:

- **Веса** — симметричная квантизация (распределение весов почти всегда симметрично около 0).
- **Активации после ReLU** — асимметричная unsigned (все значения $\geq 0$, используется UINT8).
- **Активации в трансформерах** (после LayerNorm, в residual stream) — асимметричная signed.

---

## Глава 5. Post-Training Quantization (PTQ)

### 5.1. Идея PTQ

**Post-Training Quantization** — квантизация уже обученной модели без её дообучения. Это самый практичный подход: не требует доступа к полному обучающему набору и большим вычислительным ресурсам.

Базовый процесс PTQ выглядит так:

1. Загрузить обученную модель в FP32 или FP16.
2. Прогнать через неё калибровочный набор (обычно 128–1024 примера).
3. Собрать статистики активаций для каждого слоя (мин, макс, перцентили, гистограммы).
4. Вычислить параметры квантизации $s, z$ для весов и активаций.
5. Заменить веса на квантованные значения, сохранить $s, z$.
6. Проверить точность модели.

### 5.2. Round-To-Nearest (RTN)

Простейший метод: каждый вес независимо округляется до ближайшего значения сетки.

$$\hat{w}_i = s \cdot \mathrm{round}\!\left(\frac{w_i}{s}\right)$$

Несмотря на тривиальность, RTN — это сильный baseline для 8-битной квантизации, и для многих моделей его точность приемлема. Но для INT4 и ниже RTN даёт значительные потери, что мотивирует более сложные методы.

### 5.3. GPTQ

GPTQ (*Frantar et al., 2022*) — один из наиболее влиятельных методов PTQ для LLM. Опирается на классический алгоритм OBQ (Optimal Brain Quantization) и связан с приближённой минимизацией ошибки реконструкции послойного выхода:

$$\min_{\hat{W}} \| W X - \hat{W} X \|_F^2$$

где $X$ — матрица входных активаций калибровочной выборки.

#### Ключевая идея

При квантизации одного веса возникает ошибка, и эту ошибку можно компенсировать корректировкой соседних, ещё не квантованных весов. GPTQ делает это в замкнутой форме через обратную матрицу Гессе ошибки:

$$H = 2 X X^\top$$

Для уже выбранного квантованного значения $\hat{w}_q$ поправка для всех оставшихся весов $F$:

$$\Delta w_F = -\frac{w_q - \hat{w}_q}{[H^{-1}]_{qq}} \cdot [H^{-1}]_{F, q}$$

После каждой итерации обновляется $H^{-1}$ через формулу Шермана–Моррисона, что даёт сложность $O(d^3)$ на слой.

#### Свойства GPTQ

- Высокая точность при INT4 (потеря perplexity обычно <1% для 7B+ моделей).
- Не требует переобучения, только калибровочные данные (~128 последовательностей).
- Квантизация 70B-модели занимает несколько часов на одной GPU.
- Стандарт де-факто для INT4 LLM-инференса.

### 5.4. AWQ (Activation-aware Weight Quantization)

AWQ (*Lin et al., 2023*) строится на наблюдении: не все веса в LLM одинаково важны. Веса, через которые проходят **активации с большой амплитудой**, критичны для качества модели — даже малая ошибка в них сильно искажает выход.

#### Алгоритм

1. Прогнать калибровочные данные и определить «важные» каналы по магнитудам активаций.
2. Применить масштабирование:

$$y = W x = (W \cdot \mathrm{diag}(s)) \cdot (\mathrm{diag}(s)^{-1} x) = W' x'$$

Математически это эквивалентно (произведение неизменно), но «важные» веса становятся крупнее и квантуются точнее.

3. Параметр масштаба $s$ подобрать поиском по сетке так, чтобы минимизировать:

$$s^* = \arg\min_s \| W x - Q(W \cdot \mathrm{diag}(s)) \cdot \mathrm{diag}(s)^{-1} x \|^2$$

AWQ конкурирует с GPTQ по точности при INT4, часто работает лучше при экстремально низкой разрядности, и проще в реализации.

### 5.5. Сравнение PTQ-методов

| Метод | Время калибровки | Точность INT8 | Точность INT4 | Сложность |
|---|---|---|---|---|
| RTN | Секунды | Высокая | Средняя | Минимальная |
| GPTQ | Часы | Высокая | Высокая | Средняя |
| AWQ | Минуты–часы | Высокая | Высокая | Средняя |
| SmoothQuant | Минуты | Очень высокая | — | Низкая |

---

## Глава 6. Quantization-Aware Training (QAT)

### 6.1. Концепция QAT

**Quantization-Aware Training** — это обучение (или дообучение) модели с учётом будущей квантизации. Идея: вставить в граф вычислений операции «псевдоквантизации» (fake quantization), которые имитируют округление весов и активаций к дискретной сетке во время forward pass, но позволяют градиентам течь как через идентичность.

Так модель адаптируется к квантизационному шуму ещё в процессе обучения и компенсирует его — результат обычно превосходит PTQ при той же разрядности, особенно ниже INT8.

### 6.2. Straight-Through Estimator (STE)

Главная техническая проблема QAT: операция округления $\mathrm{round}(\cdot)$ имеет нулевую производную почти везде и не определена в целых точках:

$$\frac{\partial}{\partial r} \mathrm{round}(r) = 0 \quad \text{почти всюду}$$

Через неё невозможно прогнать градиент.

Решение — **Straight-Through Estimator**: в forward pass применяем округление, а в backward pass «пропускаем» градиент так, будто округления не было:

$$\text{Forward:} \quad q = \mathrm{round}\!\left(\frac{r}{s}\right) + z$$

$$\text{Backward:} \quad \frac{\partial q}{\partial r} \approx \frac{1}{s} \quad \text{(вместо настоящего 0)}$$

Часто STE дополняют клиппингом — градиент проходит только если значение лежит в диапазоне квантизации:

$$\frac{\partial \hat{r}}{\partial r} = \mathbb{1}[r_{\min} \leq r \leq r_{\max}]$$

STE — это эвристика, нарушающая правила дифференцирования, но эмпирически она работает удивительно хорошо.

### 6.3. LSQ: Learned Step Size Quantization

LSQ (*Esser et al., 2020*) предлагает обучать масштаб $s$ как полноценный параметр модели. Это даёт сети возможность самой выбрать оптимальный шаг квантизационной сетки для каждого слоя.

Градиент по $s$ для значения внутри диапазона:

$$\frac{\partial \hat{r}}{\partial s} = \mathrm{round}\!\left(\frac{r}{s}\right) - \frac{r}{s}$$

А для значения, выходящего за пределы:

$$\frac{\partial \hat{r}}{\partial s} = \begin{cases} q_{\min}, & r < r_{\min} \\ q_{\max}, & r > r_{\max} \end{cases}$$

LSQ остаётся одним из лучших методов QAT для агрессивной квантизации ($\leq 4$ бит) в задачах computer vision.

### 6.4. QAT для LLM

Классический QAT с полным обучением слишком дорог для LLM. Поэтому применяется несколько облегчённых вариантов:

- **QLoRA** — обучаются только маленькие LoRA-адаптеры поверх замороженных квантованных весов (NF4):

$$W_{\text{eff}} = \mathrm{dequant}(W_{\text{NF4}}) + \alpha \cdot A B^\top$$

где $A \in \mathbb{R}^{d \times r}$, $B \in \mathbb{R}^{k \times r}$, $r \ll \min(d, k)$.

Это позволяет fine-tune-ить 65B-модели на одной 48 ГБ GPU.

- **QAT-finetune** — короткое дообучение (1–2 эпохи) с fake quantization на относительно небольшом наборе данных.
- **OmniQuant** — обучает только параметры самой квантизации ($s, z$ и сдвиги), оставляя веса замороженными. Сочетает преимущества PTQ и QAT.

### 6.5. PTQ или QAT — что выбрать?

| Критерий | PTQ | QAT |
|---|---|---|
| Доступ к обучающим данным | Нужна только калибровка | Нужен полноценный набор |
| Вычислительные ресурсы | Минимальные | Близкие к обучению |
| Точность при INT8 | Обычно достаточная | Незначительно лучше |
| Точность при INT4 и ниже | Возможны заметные потери | Существенно лучше |
| Время разработки | Часы | Дни–недели |

---

## Глава 7. Продвинутые методы для LLM

### 7.1. Проблема outlier-активаций

В трансформерах, начиная с моделей ~6B+ параметров, в активациях появляются **outliers** — отдельные размерности (каналы) с систематически большими значениями (в десятки и сотни раз превышающими типичные).

Формально: если для большинства каналов $|x_j| \lesssim \sigma$, то для outlier-каналов $|x_{j^*}| \gg \sigma$, и таких каналов очень мало (часто <1%).

Эти outlier-каналы оказываются критичными для производительности модели, но они же раздувают диапазон квантизации:

$$r_{\max} = \max_j |x_j| \approx |x_{j^*}|$$

В результате эффективное разрешение для типичных значений падает до $\sim \sigma / |x_{j^*}|$ от теоретического. Это явление впервые систематически описано в работе LLM.int8() (Dettmers et al., 2022) и стало главной проблемой квантизации LLM.

### 7.2. LLM.int8()

LLM.int8() решает проблему через смешанную точность:

1. Идентифицируются «outlier-каналы» в активациях (по порогу амплитуды, обычно $|x| > 6$).
2. Эти каналы обрабатываются в FP16, остальные — в INT8.
3. Результаты складываются:

$$y = \sum_{j \in \mathcal{O}} x_j W_{j,:}^{\text{FP16}} + \sum_{j \notin \mathcal{O}} x_j W_{j,:}^{\text{INT8}}$$

Это первый метод, позволивший запустить 175B-модель (OPT-175B) на доступном железе без потери качества. Однако смешанное вычисление неоптимально по скорости — оно даёт экономию памяти, но не ускорение.

### 7.3. SmoothQuant

SmoothQuant (*Xiao et al., 2022*) решает ту же проблему элегантнее. Outliers сложно квантовать **в активациях**, но веса легко квантуются. SmoothQuant «переносит» сложность из активаций в веса через математически эквивалентное преобразование:

$$Y = X W = \underbrace{(X \cdot \mathrm{diag}(s)^{-1})}_{\tilde{X}} \cdot \underbrace{(\mathrm{diag}(s) \cdot W)}_{\tilde{W}}$$

Вектор масштаба $s$ выбирается так, чтобы выровнять диапазоны активаций и весов:

$$s_j = \frac{\max(|X_{:,j}|)^\alpha}{\max(|W_{j,:}|)^{1-\alpha}}, \quad \alpha \in [0, 1]$$

Параметр $\alpha$ обычно равен $0.5$ (балансировка) и регулирует, какая часть «сложности» переносится на веса. Для outlier-каналов $s_j$ делается большим, что сглаживает активации, но раздувает соответствующие столбцы $W$. Поскольку $W$ можно квантовать per-channel, эта проблема решается легко.

Результат: чистая INT8-квантизация без смешанной точности, с минимальными потерями качества и реальным ускорением.

### 7.4. SpQR: Sparse-Quantized Representation

SpQR (*Dettmers et al., 2023*) — гибридный подход: основная масса весов квантуется агрессивно (например, в 3 бита), но небольшая доля «трудных» весов (~1%) хранится в высокой точности отдельно:

$$W = W_{\text{quant}}^{\text{INT3}} + W_{\text{outlier}}^{\text{FP16, sparse}}$$

Это даёт почти lossless-сжатие до ~3.5 бит на параметр.

### 7.5. QuIP и QuIP#

QuIP (*Chee et al., 2023*) — метод, использующий случайные ортогональные преобразования (incoherence processing) для распределения «важности» по координатам:

$$W' = U W V^\top, \quad U, V \in \mathcal{O}(d)$$

где $U, V$ — случайные ортогональные матрицы. Это делает распределение значений более «однородным» и устойчивым к квантизации. QuIP# (улучшенная версия) использует обучаемые кодовые таблицы (vector quantization) и достигает 2-битной квантизации с приемлемым качеством.

### 7.6. AQLM: Additive Quantization for LLMs

AQLM (*Egiazarian et al., 2024*) применяет к LLM аддитивную векторную квантизацию: веса представляются как сумма нескольких кодовых векторов из обучаемых кодовых таблиц $\{\mathcal{C}_m\}$:

$$\hat{w} = \sum_{m=1}^{M} c_m, \quad c_m \in \mathcal{C}_m$$

Это даёт качество, недостижимое для скалярной квантизации при <3 бит, но требует более сложного декодирования на инференсе.

### 7.7. BitNet b1.58

Радикальный подход: каждый вес — это $\{-1, 0, +1\}$ (тернарное представление, $\log_2 3 \approx 1.58$ бит). При обучении с нуля с таким ограничением модели достигают качества FP16-аналогов при огромном выигрыше в эффективности.

Квантизация весов в BitNet:

$$\hat{W}_{ij} = \mathrm{round}_{\text{clip}}\!\left(\frac{W_{ij}}{\gamma + \varepsilon}, -1, +1\right), \quad \gamma = \frac{1}{nm} \sum_{i,j} |W_{ij}|$$

BitNet (*Ma et al., 2024*) показывает, что 3B-модель в 1.58 бит сравнима с FP16 LLaMA 3B по perplexity и обходит её по скорости и энергопотреблению.

> 🔮 **Перспектива:** BitNet и подобные методы намекают на возможное будущее: модели, изначально спроектированные для крайне низкой разрядности, могут оказаться эффективнее любых пост-хок квантизаций.

---

## Глава 8. Квантизация активаций и KV-кэша

### 8.1. Зачем квантизовать активации?

Если квантовать только веса, инференс остаётся вычислительно тяжёлым: матричное умножение всё равно происходит в FP16, потому что активации не квантованы. Квантизация активаций позволяет использовать целочисленные тензорные ядра, что даёт реальное ускорение.

Однако активации сложнее квантовать, чем веса, по нескольким причинам:

- **Динамичность** — активации зависят от входа и могут сильно варьироваться.
- **Outliers** — особенно в трансформерах (см. главу 7).
- **Невозможна per-channel квантизация эффективно** — это требует пересчёта на каждый matmul.

### 8.2. W8A8, W4A16, W4A8 и другие обозначения

В литературе используется обозначение W$x$A$y$, где $x$ — биты на вес, $y$ — биты на активацию:

- **W8A8** — INT8 веса и INT8 активации. Классическая «полная» квантизация, ускоряет инференс.
- **W4A16** — INT4 веса, FP16 активации. Самый популярный режим для LLM-инференса (экономия памяти, без потерь точности).
- **W4A8** — INT4 веса, INT8 активации. Агрессивно, требует SmoothQuant или подобных методов.
- **W8A16** — INT8 веса, FP16 активации. Простой вариант, экономит память в 2 раза.
- **FP8** — обычно W8A8 в формате E4M3/E5M2 (NVIDIA Hopper).

### 8.3. KV-кэш и его квантизация

При авторегрессивной генерации в трансформере вычисляются проекции Key ($K$) и Value ($V$) для всех предыдущих токенов. Их кэширование (KV-cache) ускоряет генерацию, но кэш быстро растёт.

Размер KV-кэша:

$$\text{Size}_{\text{KV}} = 2 \cdot L \cdot n_h \cdot d_h \cdot n_{\text{tok}} \cdot b$$

где $L$ — число слоёв, $n_h$ — число голов внимания, $d_h$ — размерность головы, $n_{\text{tok}}$ — длина последовательности, $b$ — байты на элемент.

Для LLaMA-70B при контексте 32K токенов KV-кэш в FP16 занимает ~40 ГБ — больше, чем сами веса в INT4. Поэтому квантизация KV-кэша критична для длинных контекстов.

#### Методы квантизации KV-кэша

- **KV8** — INT8 квантизация $K$ и $V$ per-token. Снижает память в 2 раза, минимальные потери.
- **KV4** — INT4 квантизация. Заметные потери на длинных контекстах; обычно требует калибровки.
- **KVQuant** (Hooper et al., 2024) — учитывает структуру outlier-каналов в $K$ и $V$ отдельно, использует разные стратегии для них.
- **Smooth-KV** — переносит outliers через preconditioning, аналогично SmoothQuant.

> 💡 **Практика:** для большинства задач KV-кэш можно квантовать до INT8 без заметных потерь. INT4 KV-cache требует осторожности и подходит для memory-constrained сценариев.

---

## Глава 9. Аппаратные аспекты и форматы

### 9.1. Что реально ускоряет инференс?

Квантизация даёт два вида преимуществ:

1. **Снижение объёма памяти и трафика** — выигрывает любая модель, особенно если она memory-bound (LLM при batch=1).
2. **Ускорение вычислений** — выигрывает только если железо поддерживает целочисленные операции с нужной разрядностью.

Roofline-модель для memory-bound операций (matmul при малом batch):

$$T_{\text{inference}} \approx \frac{\text{Size}_{\text{weights}}}{\text{BW}_{\text{memory}}}$$

Поэтому уменьшение весов в 4 раза (FP16 → INT4) даёт почти 4× ускорение даже без целочисленных ядер.

На GPU NVIDIA целочисленные операции выполняются на Tensor Cores. Поддержка по поколениям:

| Архитектура | Год | Примеры GPU | Форматы Tensor Cores |
|---|---|---|---|
| Volta | 2017 | V100 | FP16 |
| Turing | 2018 | T4, RTX 20xx | FP16, INT8, INT4 |
| Ampere | 2020 | A100, RTX 30xx | BF16, TF32, INT8, INT4 |
| Hopper | 2022 | H100, H200 | FP8 (E4M3/E5M2), INT8 |
| Blackwell | 2024 | B100, B200, GB200 | FP4, FP6, FP8, INT8 |

### 9.2. Программные стеки

- **TensorRT** (NVIDIA) — наиболее производительный inference engine, поддерживает все форматы вплоть до FP4. Использует продвинутую калибровку с KL-divergence.
- **ONNX Runtime** — кросс-платформенный, поддерживает INT8/INT4 на CPU, GPU и специализированных акселераторах.
- **vLLM** — оптимизированный движок для LLM-инференса, нативно работает с GPTQ, AWQ, FP8.
- **llama.cpp** — C++-движок для CPU и Apple Silicon, флагман форматов GGUF (Q2_K…Q8_0).
- **bitsandbytes** — Python-библиотека, реализующая LLM.int8(), NF4, QLoRA.
- **TGI** (Hugging Face) — production-grade сервер с поддержкой GPTQ, AWQ, EETQ, FP8.

### 9.3. CPU-инференс

На CPU квантизация особенно важна, так как у CPU нет специализированных tensor cores. Инструкции AVX-512 VNNI (Intel) и SVE (ARM) реализуют целочисленное матричное умножение и дают 2–4× ускорение на INT8.

Apple Silicon (M-серия) исключительно эффективен на CPU/GPU/NPU инференсе квантованных моделей: 8B-модель в Q4 работает на M2 Pro со скоростью >30 ток/с.

---

## Глава 10. Практические рекомендации

### 10.1. Выбор стратегии

Простой decision tree для выбора подхода к квантизации:

1. **Хотите запустить большую LLM локально?** → W4A16 (GPTQ или AWQ, или GGUF Q4_K_M).
2. **Нужен максимально быстрый production-инференс на GPU?** → W8A8 (SmoothQuant) или FP8 на Hopper/Blackwell.
3. **Есть бюджет на fine-tune и важна точность при низкой разрядности?** → QAT или QLoRA с NF4.
4. **Модель для мобильного устройства?** → PTQ INT8 (поддерживается всем железом).
5. **Edge-устройство с очень ограниченной памятью?** → INT4 или ниже, обязательно QAT.

### 10.2. Как оценивать качество квантизованной модели

Базовые метрики:

- **Perplexity** (для LLM) на стандартных наборах (WikiText, C4, PTB):

$$\text{PPL} = \exp\!\left(-\frac{1}{N} \sum_{i=1}^{N} \log p(x_i \mid x_{<i})\right)$$

Должна вырасти минимально ($\leq 5\%$ при INT4).

- **Task accuracy** — точность на downstream-задачах: MMLU, HellaSwag, ARC, GSM8K (для рассуждений).
- **Human evaluation** — для генеративных моделей; квантизация может незаметно деградировать stylistic quality.
- **Latency и throughput** — ради этого всё затевалось; измеряйте на реальных нагрузках.
- **Memory footprint** — пиковое потребление VRAM/RAM.

### 10.3. Типичные проблемы и их решения

| Симптом | Вероятная причина | Решение |
|---|---|---|
| Резкое падение точности при INT8 | Outliers в активациях | SmoothQuant или per-token квантизация активаций |
| Деградация только на длинных контекстах | Квантизация KV-кэша слишком агрессивна | Перейти на FP16 KV-cache или INT8 вместо INT4 |
| Скорость не выросла | Inference engine не использует целочисленные ядра | Проверить, что используется правильный backend |
| PTQ INT4 даёт катастрофу | Калибровочный набор нерепрезентативен | Расширить calibration set, попробовать GPTQ |
| Модель путает редкие токены/языки | Embedding-слой квантизован агрессивно | Не квантовать embeddings (или хранить в FP16/INT8) |

### 10.4. Чек-лист перед деплоем

- ✅ Сравнили точность с FP16-baseline на нескольких метриках.
- ✅ Проверили inference latency и throughput на целевом железе.
- ✅ Убедились, что выбран самый эффективный backend для целевого формата.
- ✅ Протестировали edge-кейсы: длинные контексты, редкие токены, экстремальные температуры.
- ✅ Зафиксировали seed и проверили детерминизм (если требуется).
- ✅ Замерили потребление памяти при пиковой нагрузке (batch и длина контекста).

---

## Приложение А. Глоссарий

| Термин | Определение |
|---|---|
| **Activation** | Выход нейрона/слоя; данные, передаваемые от слоя к слою. |
| **AWQ** | Activation-aware Weight Quantization — метод PTQ, защищающий важные веса. |
| **BF16** | Brain Float 16 — 16-битный формат с экспонентой как у FP32. |
| **Calibration** | Прогон калибровочных данных для определения параметров квантизации. |
| **Dequantization** | Обратное преобразование $q \to \hat{r} = s(q-z)$. |
| **FP8 (E4M3/E5M2)** | 8-битные форматы с плавающей точкой. |
| **GGUF** | Файловый формат и набор квантизационных схем из llama.cpp. |
| **GPTQ** | Метод PTQ с компенсацией ошибки через гессиан. |
| **KV-cache** | Кэш Key/Value тензоров в авторегрессивных трансформерах. |
| **LLM.int8()** | Метод смешанной точности для квантизации LLM. |
| **NF4** | NormalFloat 4 — 4-битный формат для весов с гауссовым распределением. |
| **Outlier** | Экстремальное значение, нарушающее равномерность распределения. |
| **Per-channel** | Свои параметры квантизации для каждого канала. |
| **Per-group** | Свои параметры на группу элементов (обычно 32/64/128). |
| **Per-tensor** | Один набор параметров на весь тензор. |
| **PTQ** | Post-Training Quantization — квантизация после обучения. |
| **QAT** | Quantization-Aware Training — обучение с учётом квантизации. |
| **QLoRA** | Fine-tune-метод поверх NF4-квантованной базовой модели. |
| **RTN** | Round-To-Nearest — простейшая поэлементная квантизация. |
| **Scale ($s$)** | Масштабный коэффициент в формуле квантизации. |
| **SmoothQuant** | Метод переноса outliers из активаций в веса. |
| **STE** | Straight-Through Estimator — суррогатный градиент через округление. |
| **W4A16** | Обозначение: 4 бита веса, 16 бит активации. |
| **Zero-point ($z$)** | Целочисленный сдвиг, точно представляющий 0. |

---

## Приложение Б. Литература и ресурсы

### Ключевые статьи

- Jacob et al. (2018). *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*. CVPR.
- Krishnamoorthi (2018). *Quantizing deep convolutional networks for efficient inference: A whitepaper*.
- Esser et al. (2020). *Learned Step Size Quantization*. ICLR.
- Dettmers et al. (2022). *LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale*. NeurIPS.
- Xiao et al. (2022). *SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models*. ICML.
- Frantar et al. (2022). *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers*. ICLR.
- Lin et al. (2023). *AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration*. MLSys.
- Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs*. NeurIPS.
- Dettmers et al. (2023). *SpQR: A Sparse-Quantized Representation for Near-Lossless LLM Weight Compression*.
- Chee et al. (2023). *QuIP: 2-Bit Quantization of Large Language Models With Guarantees*. NeurIPS.
- Egiazarian et al. (2024). *Extreme Compression of Large Language Models via Additive Quantization*. ICML.
- Ma et al. (2024). *The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits*.
- Hooper et al. (2024). *KVQuant: Towards 10 Million Context Length LLM Inference with KV Cache Quantization*.

### Обзоры и учебные материалы

- Gholami et al. (2021). *A Survey of Quantization Methods for Efficient Neural Network Inference*.
- Nagel et al. (2021). *A White Paper on Neural Network Quantization*. Qualcomm AI Research.
- Zhu et al. (2023). *A Survey on Model Compression for Large Language Models*.

### Программные библиотеки

- **bitsandbytes** — github.com/bitsandbytes-foundation/bitsandbytes
- **AutoGPTQ** — github.com/AutoGPTQ/AutoGPTQ
- **AutoAWQ** — github.com/casper-hansen/AutoAWQ
- **llama.cpp** — github.com/ggerganov/llama.cpp
- **vLLM** — github.com/vllm-project/vllm
- **Hugging Face Optimum** — github.com/huggingface/optimum
- **TensorRT-LLM** — github.com/NVIDIA/TensorRT-LLM